# Caption octopus clips with Qwen3-VL-30B (vLLM / Colab GPU)

Captions every clip in **`octopus_clips_verified/`** and writes the caption back into
**`octopus_clips_verified.json`** under a new `caption` field on each clip entry.

Based on `exp23_caption_colab.ipynb`, but wired to the current pipeline:
- Source of truth is the index JSON (`octopus_clips_verified.json`), not a flat clip dir.
- For each clip entry **without** a `caption`, sample frames across the 20s clip, ask
  Qwen3-VL-30B for **one caption for the whole clip** (or `not present` if no octopus),
  then store it on that entry.
- **Resumable**: entries that already have a `caption` are skipped; the JSON is saved
  after every clip, so it is safe to interrupt.

Uses **Qwen3-VL-30B-A3B-Instruct (AWQ Int4)** via **vLLM** — the local Qwen2-VL-2B was too
weak (called the octopus a "fish"). Needs a GPU (A100-40GB).

> **Runtime → Change runtime type → A100 GPU.** First model load downloads ~17 GB.

## 1. Install dependencies

In [ ]:
# Remove Colab's cu128 torch FIRST so vLLM installs the exact torch it was built against
# (otherwise `from vllm import LLM` fails with libcudart.so.13 not found). See exp23 for the why.
!pip uninstall -y -q torch torchvision torchaudio torchao 2>/dev/null
!pip install -q -U vllm qwen-vl-utils
!apt-get -qq install -y ffmpeg >/dev/null

print("Installed.")
print("NOW: Runtime -> Restart session, then run from the CONFIG cell (skip this install cell).")
print("vLLM cannot be imported in the same kernel that just reinstalled torch.")

In [ ]:
import torch
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

## 2. Config

In [ ]:
from pathlib import Path

MODEL        = "QuantTrio/Qwen3-VL-30B-A3B-Instruct-AWQ"   # AWQ Int4 30B-A3B MoE (~17 GB)
#   "Qwen/Qwen3-VL-8B-Instruct"  - dense bf16, ~18 GB, fits anything (lower quality)

FRAME_FPS     = 0.5     # 1 frame / 2s -> ~10 frames per 20s clip
MAX_TOKENS    = 200
MAX_MODEL_LEN = 8192
GPU_MEM_UTIL  = 0.92
IMAGE_LIMIT   = 16

# The clip index (one entry per clip; we add a `caption` field). On Colab this is the
# local copy you unzip below; the save-back cell copies it back to Drive.
INDEX_JSON  = Path("octopus_clips_verified.json")
# Where the octopus_clips_verified/ folder lives (clips referenced by each entry's clip_path).
CLIPS_ROOT  = Path("octopus_clips_verified")
CAPTION_KEY = "caption"          # field written onto each clip entry

## 3. Get the data into Colab

Put `octopus_clips_verified.zip` (the clips folder) and `octopus_clips_verified.json`
in **`MyDrive/GSOC-Catrobat/`**, then run the Drive cell.

In [ ]:
# === Option B: Google Drive  (recommended) ===
from google.colab import drive
drive.mount("/content/drive")

import zipfile, shutil
DRIVE_ROOT = Path("/content/drive/MyDrive/GSOC-Catrobat")

with zipfile.ZipFile(DRIVE_ROOT / "octopus_clips_verified.zip") as z:
    z.extractall(".")                              # creates ./octopus_clips_verified/
shutil.copy(DRIVE_ROOT / "octopus_clips_verified.json", INDEX_JSON)

n_clips = len(list(CLIPS_ROOT.rglob("*.mp4")))
print(f"{n_clips} clips under {CLIPS_ROOT}/ | index: {INDEX_JSON}")

In [ ]:
# # === Option A: Zip upload  (SKIP if you used Drive) ===
# import zipfile, shutil
# from google.colab import files
# up = files.upload()   # select octopus_clips_verified.zip and octopus_clips_verified.json
# for name in up:
#     if name.endswith(".zip"):
#         with zipfile.ZipFile(name) as z: z.extractall(".")
#     elif name.endswith(".json"):
#         shutil.move(name, INDEX_JSON)
# print(len(list(CLIPS_ROOT.rglob("*.mp4"))), "clips ready")

## 4. Load the model

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoProcessor

print(f"Loading {MODEL} with vLLM ...  (first run downloads ~17 GB)")
llm = LLM(
    model=MODEL,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEM_UTIL,
    limit_mm_per_prompt={"image": IMAGE_LIMIT},
    dtype="auto",
    trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
sampling_params = SamplingParams(temperature=0.0, max_tokens=MAX_TOKENS)
print("Model ready (vLLM).")

## 5. Prompt

One caption for the whole clip (frames passed in order), or `not present` if the octopus
isn't visible in any frame.

In [ ]:
def build_prompt() -> str:
    return (
        "These frames are sampled in order from a single short aquarium security-camera clip. "
        "The subject is Nity, an octopus (Octopus vulgaris). "
        "Octopuses change color, extend arms, hide in dens, manipulate objects, and interact with humans.\n\n"
        "First decide whether the octopus appears in ANY frame of the clip.\n"
        "- If the octopus is NOT visible in any frame, respond with EXACTLY:\n"
        "  CAPTION: not present\n"
        "- If the octopus IS visible, write ONE caption describing what it does across the whole clip "
        "(its movement, posture, arm position, color, and anything it touches or interacts with).\n\n"
        "Respond in EXACTLY this format, nothing else:\n"
        "CAPTION: <one sentence for the whole clip>"
    )

def parse_caption(text: str) -> str:
    for line in text.splitlines():
        s = line.strip()
        if s.upper().startswith("CAPTION:"):
            return s[len("CAPTION:"):].strip().strip("'\"")
    return text.strip().strip("'\"")

## 6. Frame sampling + inference helpers

In [ ]:
import subprocess, tempfile
from qwen_vl_utils import process_vision_info

MAX_PIXELS = 512 * 512

def resolve_clip(entry) -> Path:
    """Locate the local clip file for an index entry (clip_path is repo-relative)."""
    cp = entry["clip_path"]
    rel = cp.split("octopus_clips_verified/", 1)[-1]   # <date>/<seg>/<name>.mp4
    return CLIPS_ROOT / rel

def extract_frames(clip_path: Path, tmpdir: str) -> list:
    pattern = str(Path(tmpdir) / "f_%03d.jpg")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(clip_path),
                    "-vf", f"fps={FRAME_FPS}", "-q:v", "2", pattern], check=True)
    return sorted(str(p) for p in Path(tmpdir).glob("f_*.jpg"))

def caption_clip(frame_paths: list, prompt: str) -> str:
    content = [{"type": "image", "image": p, "max_pixels": MAX_PIXELS} for p in frame_paths]
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    out = llm.generate({"prompt": text, "multi_modal_data": {"image": image_inputs}},
                       sampling_params=sampling_params, use_tqdm=False)
    return out[0].outputs[0].text.strip()

## 7. Run captioning (resumable)

Writes each caption onto its entry in `octopus_clips_verified.json` and saves after every
clip. Re-running skips entries that already have a `caption`.

In [ ]:
import json
from datetime import datetime

prompt = build_prompt()
index  = json.load(open(INDEX_JSON))
clips  = index["clips"]

todo = [c for c in clips if not c.get(CAPTION_KEY)]
print(f"{len(clips)} clips total, {len(todo)} without a caption\n" + "-" * 60)

done_n = 0
for i, entry in enumerate(todo):
    clip_path = resolve_clip(entry)
    print(f"[{i+1}/{len(todo)}] {entry['clip_path']}", flush=True)
    if not clip_path.exists():
        print(f"  ! missing file: {clip_path}"); continue
    with tempfile.TemporaryDirectory() as tmp:
        try:
            frames = extract_frames(clip_path, tmp)
            if not frames:
                print("  no frames"); continue
            raw = caption_clip(frames, prompt)
        except Exception as e:
            print(f"  failed: {e}"); continue

    caption = parse_caption(raw)
    entry[CAPTION_KEY]   = caption
    entry["captioned_at"] = datetime.now().isoformat(timespec="seconds")
    entry["caption_model"] = MODEL.split("/")[-1]
    print(f"  caption: {caption}", flush=True)

    # save after every clip -> resumable
    with open(INDEX_JSON, "w") as f:
        json.dump(index, f, indent=2)
    done_n += 1

print("-" * 60 + f"\nDone. captioned {done_n} clips this run.")
print(f"with caption now: {sum(1 for c in clips if c.get(CAPTION_KEY))}/{len(clips)}")

## 8. Save the JSON back

Copies the updated `octopus_clips_verified.json` back to Drive so it persists. Then drop it
into `data/octopus_clips_verified.json` in the repo.

In [ ]:
import shutil
if Path("/content/drive/MyDrive").exists():
    shutil.copy(INDEX_JSON, DRIVE_ROOT / INDEX_JSON.name)
    print(f"Copied back to Drive: {DRIVE_ROOT / INDEX_JSON.name}")
else:
    from google.colab import files
    files.download(str(INDEX_JSON))